# 数据准备
拿到手的数据是一个excel

In [1]:
import pandas as pd

df_train = pd.read_csv('training.csv')

In [2]:
df_train

,student_id,article_id,summary_text,score
0,56d9a67c9d09,3b9047,"the article by UShistory. Org ""Egypt social s...",0.846025
1,89a277adc42c,3b9047,In the upper class There was the royal famile...,-0.456956
2,96d4353c6615,3b9047,In the Egyptian government system only the nob...,0.376374
3,dd4b17567486,39c16e,Aristotle describes many elements of an ideal ...,1.690740
4,91bcf10857b6,ebad26,They would use chemicals to get rid of mold an...,0.050689
...,...,...,...,...
5727,85899a7b2631,3b9047,The social classes were involved due to how th...,-0.970237
5728,b83d40cf4b03,ebad26,Jonas had told them how the meat that was take...,-1.547163
5729,b97ee0bdafbd,ebad26,"In The Jungle, when meat was spoiled it could ...",1.038163
5730,bf2bed944599,39c16e,One element is that a tragedy should be arrang...,0.531368


# 从原文本直接提取一些文本特征

In [3]:
import textstat
import re
# 1. 定义特征工程函数
def extract_features(text):
    """
    从一段文本中提取特征
    """
    features = {}

    # 基础统计特征
    words = text.split()
    sentences = re.split(r'[.?!]+', text)
    sentences = [s for s in sentences if len(s) > 0] # 移除空句子

    num_chars = len(text)
    num_words = len(words)
    num_sentences = len(sentences) if len(sentences) > 0 else 1
    num_unique_words = len(set(w.lower() for w in words))

    features['num_chars'] = num_chars
    features['num_words'] = num_words
    features['num_sentences'] = num_sentences
    features['avg_word_length'] = num_chars / num_words if num_words > 0 else 0
    features['avg_sentence_length'] = num_words / num_sentences if num_sentences > 0 else 0
    features['unique_word_ratio'] = num_unique_words / num_words if num_words > 0 else 0

    # 使用 textstat 库提取可读性指标
    try:
        # Flesch阅读简易度：得分越高，文本越容易理解（满分100分表示最容易理解）
        features['flesch_reading_ease'] = textstat.flesch_reading_ease(text)
        # Gunning Fog指数：得分越高，文本越难理解（通常对应更高的教育水平要求）
        features['gunning_fog'] = textstat.gunning_fog(text)
        # SMOG指数：基于多音节词数量估算文本复杂度（常用于医疗/法律等专业领域）
        features['smog_index'] = textstat.smog_index(text)
        # 自动可读性指数：基于字符和单词数量估算理解文本所需的年级水平
        features['automated_readability_index'] = textstat.automated_readability_index(text)
        # Coleman-Liau指数：通过字符数和句子数计算美国教育年级水平
        features['coleman_liau_index'] = textstat.coleman_liau_index(text)
        # Linsear Write公式：用于军事和政府文档，估算阅读难度对应的年级水平
        features['linsear_write_formula'] = textstat.linsear_write_formula(text)
        # Dale-Chall可读性得分：基于普通读者认识的单词列表评估难度
        features['dale_chall_readability_score'] = textstat.dale_chall_readability_score(text)
        # 音节计数：计算文本中总的音节数量
        features['syllable_count'] = textstat.syllable_count(text)
        # 词汇计数：计算文本中唯一单词的数量（去除标点但保留数字）
        features['lexicon_count'] = textstat.lexicon_count(text)
        # 多音节词计数：计算包含3个或更多音节的单词数量
        features['polysyllabcount'] = textstat.polysyllabcount(text)
    except Exception as e:
        # 如果textstat处理失败（例如文本太短），则填充为0
        print(f"Textstat error: {e}. Filling features with 0.")
        for key in ['flesch_reading_ease', 'gunning_fog', 'smog_index', 'automated_readability_index',
                    'coleman_liau_index', 'linsear_write_formula', 'dale_chall_readability_score',
                    'syllable_count', 'lexicon_count', 'polysyllabcount']:
            features[key] = 0

    return features

In [4]:
# 3. 应用特征工程
feature_list = []
for text in df_train['summary_text']:
    feature_list.append(extract_features(text))

# 4. 将特征列表转换为DataFrame
df_features = pd.DataFrame(feature_list)

# # 5.将新特征与原DataFrame合并（可选，但通常有益）
# df_train = pd.concat([df_train, df_features], axis=1)

In [5]:
df_features

,num_chars,num_words,num_sentences,avg_word_length,avg_sentence_length,unique_word_ratio,flesch_reading_ease,gunning_fog,smog_index,automated_readability_index,coleman_liau_index,linsear_write_formula,dale_chall_readability_score,syllable_count,lexicon_count,polysyllabcount
0,595,103,7,5.776699,14.714286,0.679612,68.221683,8.543706,9.606745,8.145354,9.425743,7.285714,9.667603,148,101,9
1,246,43,3,5.720930,14.333333,0.604651,76.207597,8.524031,8.841846,7.862713,8.976744,7.166667,9.121154,59,43,3
2,452,79,3,5.721519,26.333333,0.746835,49.458565,14.077637,14.068176,14.034641,9.946835,16.833333,8.940102,122,79,11
3,713,123,7,5.796748,17.571429,0.609756,50.063415,13.207433,12.785404,9.948397,9.817886,8.714286,10.413246,202,123,20
4,363,70,4,5.185714,17.500000,0.657143,62.842647,15.364706,10.125757,15.284714,7.297059,18.500000,8.805988,88,68,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5727,203,38,3,5.342105,12.666667,0.789474,87.115175,7.171930,7.793538,5.478596,6.742105,6.000000,8.420030,48,38,2
5728,198,41,1,4.829268,41.000000,0.829268,62.049268,16.400000,3.129100,17.220732,5.404878,20.500000,6.825466,50,41,0
5729,923,172,7,5.366279,24.571429,0.598837,65.258842,15.317427,10.793553,16.262558,7.823392,18.666667,8.564691,216,171,9
5730,417,77,3,5.415584,25.666667,0.636364,51.136580,17.539394,16.322122,12.261905,8.205195,18.166667,8.805800,118,77,16


# 数据清洗

In [6]:
import re
import nltk
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer



def text_cleaner(text):
    # # 1. 移除 HTML 标签
    # text = BeautifulSoup(text, "html.parser").get_text()
    #
    # # 2. 移除 HTML 实体（如 &amp; → &，&lt; → <）
    # text = re.sub(r'&[a-z]+;', ' ', text)
    #
    # 3. 统一小写
    text = text.lower()
    #
    # # 4. 处理连字符：
    # #    - 保留复合词中的连字符（如 far-reaching）
    # #    - 去掉连接短语中的连字符（如 u.s.-japan → u.s. japan）
    # text = re.sub(r'(?<=\b[a-zA-Z])- (?=[a-zA-Z]\b)', '', text)  # 保留复合词
    # text = re.sub(r'(?<=\b[a-zA-Z])-(?=[a-zA-Z]\b)', ' ', text)  # 处理 U.S.-Japan 类连接词
    #
    # # 5. 仅保留字母、空格和句号
    # text = re.sub(r'[^a-zA-Z\s.]', '', text)
    #
    # # 6. 合并多余空格
    # text = re.sub(r'\s+', ' ', text).strip()
    #
    # 7. 分词
    tokens = word_tokenize(text)
    #
    # 8. 词形还原（Lemmatization）
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word, pos='v') for word in tokens]  # 还原为动词基本形式

    # 9. 去停用词
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    return tokens

## 另一种数据清洗方式
可以洗html

In [ ]:
from bs4 import BeautifulSoup
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords
import string

# 词性标签转换函数（核心改进）
def get_wordnet_pos(tag):
    """将nltk的POS标签转换为wordnet的格式"""
    if tag.startswith('N'):
        return wn.NOUN
    elif tag.startswith('V'):
        return wn.VERB
    elif tag.startswith('J'):
        return wn.ADJ
    elif tag.startswith('R'):
        return wn.ADV
    return wn.NOUN  # 默认按名词处理

def text_cleaner(text):
    # 1. 去除HTML标签
    text = BeautifulSoup(text, 'html.parser').get_text()

    # 2. 统一小写
    text = text.lower()

    # 3. 分词
    tokens = word_tokenize(text)

    # 4. 词性标注（新增步骤）
    pos_tags = nltk.pos_tag(tokens)

    # 5. 带词性的词形还原（核心改进）
    lemmatizer = WordNetLemmatizer()
    tokens = [
        lemmatizer.lemmatize(word, pos=get_wordnet_pos(tag))
        for (word, tag) in pos_tags
    ]

    # 6. 去停用词
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # +++++++++++++ 新增步骤：去除标点符号 +++++++++++++
    # 方法1：过滤纯标点符号的token（如逗号、句号）
    punctuation = set(string.punctuation)
    tokens = [word for word in tokens if word not in punctuation]

    # 方法2：过滤所有包含非字母字符的token（同时去除数字和混合字符）
    tokens = [word for word in tokens if word.isalpha()]

    return tokens


## 把数据清洗应用到数据集上
这里形成的df就是有一列clean_text（列表形式，清洗完的单词）和jointed_text（string形式，清洗完的单词用空格连接）的df。TFIDF用的是joined_text，Word2Vec用的是cleaned_text

In [7]:
# 应用到数据集
df_train['cleaned_text'] = df_train['summary_text'].apply(text_cleaner)
df_train['cleaned_text']

0       [article, ushistory, ., org, ``, egypt, social...
1       [upper, class, royal, familes, goverments, lan...
2       [egyptian, government, system, nobles, allow, ...
3       [aristotle, describe, many, elements, ideal, t...
4       [would, use, chemicals, get, rid, mold, discol...
                              ...                        
5727    [social, class, involve, due, prisoners, war, ...
5728    [jonas, tell, meat, take, pickle, would, often...
5729    [jungle, ,, meat, spoil, could, chop, piece, s...
5730    [one, element, tragedy, arrange, way, make, re...
5731    [would, mix, spoil, meat, good, meat, ., would...
Name: cleaned_text, Length: 5732, dtype: object

In [8]:
df_train['joined_text'] = df_train['cleaned_text'].apply(' '.join) #这里处理后的joined_text列就是最后用到的文本数据
df_train['joined_text']

0       article ushistory . org `` egypt social status...
1       upper class royal familes goverments landowner...
2       egyptian government system nobles allow govern...
3       aristotle describe many elements ideal tragedy...
4       would use chemicals get rid mold discolouratio...
                              ...                        
5727    social class involve due prisoners war lead sl...
5728    jonas tell meat take pickle would often find s...
5729    jungle , meat spoil could chop piece sausage ,...
5730    one element tragedy arrange way make reader/au...
5731    would mix spoil meat good meat . would pour ch...
Name: joined_text, Length: 5732, dtype: object

In [9]:
df_train

,student_id,article_id,summary_text,score,cleaned_text,joined_text
0,56d9a67c9d09,3b9047,"the article by UShistory. Org ""Egypt social s...",0.846025,"[article, ushistory, ., org, ``, egypt, social...",article ushistory . org `` egypt social status...
1,89a277adc42c,3b9047,In the upper class There was the royal famile...,-0.456956,"[upper, class, royal, familes, goverments, lan...",upper class royal familes goverments landowner...
2,96d4353c6615,3b9047,In the Egyptian government system only the nob...,0.376374,"[egyptian, government, system, nobles, allow, ...",egyptian government system nobles allow govern...
3,dd4b17567486,39c16e,Aristotle describes many elements of an ideal ...,1.690740,"[aristotle, describe, many, elements, ideal, t...",aristotle describe many elements ideal tragedy...
4,91bcf10857b6,ebad26,They would use chemicals to get rid of mold an...,0.050689,"[would, use, chemicals, get, rid, mold, discol...",would use chemicals get rid mold discolouratio...
...,...,...,...,...,...,...
5727,85899a7b2631,3b9047,The social classes were involved due to how th...,-0.970237,"[social, class, involve, due, prisoners, war, ...",social class involve due prisoners war lead sl...
5728,b83d40cf4b03,ebad26,Jonas had told them how the meat that was take...,-1.547163,"[jonas, tell, meat, take, pickle, would, often...",jonas tell meat take pickle would often find s...
5729,b97ee0bdafbd,ebad26,"In The Jungle, when meat was spoiled it could ...",1.038163,"[jungle, ,, meat, spoil, could, chop, piece, s...","jungle , meat spoil could chop piece sausage ,..."
5730,bf2bed944599,39c16e,One element is that a tragedy should be arrang...,0.531368,"[one, element, tragedy, arrange, way, make, re...",one element tragedy arrange way make reader/au...


# 文本向量化

## TFIDF

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 创建TF-IDF模型[1,3](@ref)
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,        # 保留前1000个重要特征
    ngram_range=(1,2),        # 包含1-2元语法（就是把一个词当成一个特征还是两个词，比如new york得2-gram才能识别）
    min_df=2                  # 忽略文档频率<2的词
)

# 生成TF-IDF矩阵
tfidf_matrix = tfidf_vectorizer.fit_transform(df_train['joined_text'])

# 转换为DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

In [11]:
tfidf_df

,10,11,11 12,11 say,12,13,13 say,1906,1967,200,...,write wish,writer,wrong,year,yearly,yearly harvest,yet,yet spite,yet whose,young
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5727,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5728,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5729,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 构建X和Y

In [12]:
Y = df_train.loc[:,['score']]
Y

,score
0,0.846025
1,-0.456956
2,0.376374
3,1.690740
4,0.050689
...,...
5727,-0.970237
5728,-1.547163
5729,1.038163
5730,0.531368


In [13]:
X = pd.concat([df_train['article_id'],df_features, tfidf_df], axis=1)
X

,article_id,num_chars,num_words,num_sentences,avg_word_length,avg_sentence_length,unique_word_ratio,flesch_reading_ease,gunning_fog,smog_index,...,write wish,writer,wrong,year,yearly,yearly harvest,yet,yet spite,yet whose,young
0,3b9047,595,103,7,5.776699,14.714286,0.679612,68.221683,8.543706,9.606745,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3b9047,246,43,3,5.720930,14.333333,0.604651,76.207597,8.524031,8.841846,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3b9047,452,79,3,5.721519,26.333333,0.746835,49.458565,14.077637,14.068176,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,39c16e,713,123,7,5.796748,17.571429,0.609756,50.063415,13.207433,12.785404,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ebad26,363,70,4,5.185714,17.500000,0.657143,62.842647,15.364706,10.125757,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5727,3b9047,203,38,3,5.342105,12.666667,0.789474,87.115175,7.171930,7.793538,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5728,ebad26,198,41,1,4.829268,41.000000,0.829268,62.049268,16.400000,3.129100,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5729,ebad26,923,172,7,5.366279,24.571429,0.598837,65.258842,15.317427,10.793553,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5730,39c16e,417,77,3,5.415584,25.666667,0.636364,51.136580,17.539394,16.322122,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
X = pd.get_dummies(X, columns=['article_id'])
X

,num_chars,num_words,num_sentences,avg_word_length,avg_sentence_length,unique_word_ratio,flesch_reading_ease,gunning_fog,smog_index,automated_readability_index,...,yearly,yearly harvest,yet,yet spite,yet whose,young,article_id_39c16e,article_id_3b9047,article_id_814d6b,article_id_ebad26
0,595,103,7,5.776699,14.714286,0.679612,68.221683,8.543706,9.606745,8.145354,...,0.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False
1,246,43,3,5.720930,14.333333,0.604651,76.207597,8.524031,8.841846,7.862713,...,0.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False
2,452,79,3,5.721519,26.333333,0.746835,49.458565,14.077637,14.068176,14.034641,...,0.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False
3,713,123,7,5.796748,17.571429,0.609756,50.063415,13.207433,12.785404,9.948397,...,0.0,0.0,0.0,0.0,0.0,0.0,True,False,False,False
4,363,70,4,5.185714,17.500000,0.657143,62.842647,15.364706,10.125757,15.284714,...,0.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5727,203,38,3,5.342105,12.666667,0.789474,87.115175,7.171930,7.793538,5.478596,...,0.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False
5728,198,41,1,4.829268,41.000000,0.829268,62.049268,16.400000,3.129100,17.220732,...,0.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True
5729,923,172,7,5.366279,24.571429,0.598837,65.258842,15.317427,10.793553,16.262558,...,0.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True
5730,417,77,3,5.415584,25.666667,0.636364,51.136580,17.539394,16.322122,12.261905,...,0.0,0.0,0.0,0.0,0.0,0.0,True,False,False,False


# LightGBM K折交叉验证

In [15]:
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import gc # 垃圾回收库，用于管理内存
import numpy as np

# ------------------------------------------------------------------------------------
# 2. 模型与交叉验证设置 (Model and Cross-Validation Setup)
# ------------------------------------------------------------------------------------

# 设置K折交叉验证
# n_splits: K的值，表示要将数据分成多少份。常用的值为5或10。
# shuffle=True: 在分割数据前先打乱顺序，这对于保证每一折数据的分布相似性很重要。
# random_state: 设置一个固定的随机种子，以确保每次运行代码时，数据的分割方式都一样，便于复现结果。
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# 初始化 LightGBM 回归模型 (LGBMRegressor)
# 这里设置了一些常用的超参数，您可以根据需要进行调整。
# 详细的优化建议见代码末尾的注释。
lgbm = lgb.LGBMRegressor(
    objective='regression_l1', # 目标函数, 'regression_l1' (MAE) 对异常值更鲁棒, 也可使用 'regression_l2' (MSE)
    metric='rmse',             # 评估指标, 使用均方根误差 (Root Mean Squared Error)
    n_estimators=4000,         # 决策树的数量。这是一个可以调整的关键参数。
    learning_rate=0.005,        # 学习率，控制每次迭代中模型的更新步长。
    n_jobs=-1,                 # 使用所有可用的CPU核心进行并行计算。
    random_state=42,           # 随机种子，保证模型训练的可复现性。
    # --- 以下为防止过拟合和提高性能的参数 ---
    num_leaves=31,             # 每棵树的最大叶子节点数。
    max_depth=-1,              # 每棵树的最大深度, -1表示不限制。
    colsample_bytree=0.8,      # 训练每棵树时，随机选择80%的特征。
    subsample=0.8,             # 训练每棵树时，随机选择80%的样本。
    reg_alpha=0.1,             # L1正则化系数。
    reg_lambda=0.1             # L2正则化系数。
)

In [16]:
# ------------------------------------------------------------------------------------
# 3. 执行交叉验证与模型评估 (Cross-Validation Execution and Evaluation)
# ------------------------------------------------------------------------------------

# 存储每一折的最佳迭代次数
optimal_rounds = []

# 创建一个列表来存储每一折的评估分数
oof_predictions = np.zeros(X.shape[0]) # 用于存储袋外（Out-of-Fold）预测
rmse_scores = []

print("开始进行K折交叉验证...")
# 循环遍历KFold生成的每一折数据
for fold, (train_index, val_index) in enumerate(kf.split(X, Y)):
    print(f"--- 第 {fold+1}/{N_SPLITS} 折 ---")

    # 1. 数据分割
    # 根据当前折的索引，切分训练集和验证集
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    Y_train, Y_val = Y.iloc[train_index], Y.iloc[val_index]

    # 2. 模型训练
    # 使用 .fit() 方法训练模型
    # eval_set: 提供验证集，用于监控模型性能和实现早停。
    # callbacks: LightGBM的回调函数列表。lgb.early_stopping 是一个早停回调。
    #   - stopping_rounds: 如果验证集上的评估指标在N轮内没有改善，就提前停止训练。这能有效防止过拟合，并找到最佳的树数量。
    #   - verbose=False: 不打印早停的详细过程。
    model = lgbm.fit(X_train, Y_train,
                     eval_set=[(X_val, Y_val)],
                     eval_metric='rmse',
                     callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])

    # model.best_iteration_ 属性记录了早停时最佳的迭代次数
    print(f"第 {fold+1} 折 - 最佳迭代次数: {model.best_iteration_}")
    optimal_rounds.append(model.best_iteration_)


    # 3. 模型预测
    # 对当前折的验证集进行预测
    val_preds = model.predict(X_val)
    oof_predictions[val_index] = val_preds

    # 4. 性能评估
    # 计算当前折的均方根误差 (RMSE)
    # RMSE可以直观地反映预测值与真实值之间的差距，单位与目标变量相同。
    rmse = np.sqrt(mean_squared_error(Y_val, val_preds))
    rmse_scores.append(rmse)
    print(f"第 {fold+1} 折的 RMSE: {rmse:.4f}")

    # 释放内存
    del X_train, X_val, Y_train, Y_val, model
    gc.collect()

print("\n交叉验证完成。")
# ------------------------------------------------------------------------------------
# 4. 输出最终评估结果 (Final Evaluation Results)
# ------------------------------------------------------------------------------------

# 计算并打印所有折的平均RMSE和标准差
# 平均RMSE: 模型在不同数据子集上的平均表现，是衡量模型整体性能的核心指标。
# RMSE标准差: 反映模型性能在不同数据子集上的稳定性。如果标准差很大，说明模型可能对数据的划分方式很敏感。
print(f"\n{N_SPLITS}折交叉验证的平均 RMSE: {np.mean(rmse_scores):.4f}")
print(f"{N_SPLITS}折交叉验证的 RMSE 标准差: {np.std(rmse_scores):.4f}")


# 计算平均最佳迭代次数，并增加一点余量（例如10%），然后取整
# 这样做是因为在更大的数据集（全部数据）上训练时，模型可能需要更多的迭代。
final_n_estimators = int(np.mean(optimal_rounds) * 1.1)
print(f"\n计算出的平均最佳迭代次数为: {int(np.mean(optimal_rounds))}")
print(f"用于最终模型的 n_estimators 参数为: {final_n_estimators}")

开始进行K折交叉验证...
--- 第 1/5 折 ---
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012133 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 65872
[LightGBM] [Info] Number of data points in the train set: 4585, number of used features: 1984
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score -0.093814
第 1 折 - 最佳迭代次数: 3365
第 1 折的 RMSE: 0.4473
--- 第 2/5 折 ---
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.021116 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 65601
[LightGBM] [Info] Number of data points in the train set: 4585, num

# 全量模型构建

In [17]:
# ------------------------------------------------------------------------------------
# 5. 训练最终模型并进行预测 (Train Final Model and Predict)
# ------------------------------------------------------------------------------------
print("\n--- 开始使用全部数据训练最终模型 ---")

# 初始化最终模型
# 使用上面计算出的 final_n_estimators 作为决策树的数量
# 其他参数保持与交叉验证时一致，以保证模型的行为相似
final_model = lgb.LGBMRegressor(
    objective='regression_l1',
    metric='rmse',
    n_estimators=final_n_estimators, # 使用我们计算出的最佳轮数
    learning_rate=0.005,
    n_jobs=-1,
    random_state=42,
    num_leaves=31,
    max_depth=-1,
    colsample_bytree=0.8,
    subsample=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1
)

# 在全部的 X 和 Y 数据上训练模型
# 注意：这里不再需要 eval_set 和 callbacks，因为我们已经确定了 n_estimators
final_model.fit(X, Y)

print("最终模型训练完成。")



--- 开始使用全部数据训练最终模型 ---
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.034349 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 82541
[LightGBM] [Info] Number of data points in the train set: 5732, number of used features: 2328
[LightGBM] [Info] Start training from score -0.093814
最终模型训练完成。


In [18]:
import joblib

# 保存模型
joblib.dump(final_model, 'LightGBM.pkl')
print('保存模型成功')

保存模型成功


# 处理需预测数据集

In [19]:
df_new = pd.read_csv('test.csv')

## 提取统计特征

In [20]:
feature_list = []
for text in df_new['summary_text']:
    feature_list.append(extract_features(text))

# 4. 将特征列表转换为DataFrame
df_features_new = pd.DataFrame(feature_list)

## TFIDF特征提取

### 数据清洗

In [21]:
# 应用到数据集
df_new['cleaned_text'] = df_new['summary_text'].apply(text_cleaner)
df_new['joined_text'] = df_new['cleaned_text'].apply(' '.join) #这里处理后的joined_text列就是最后用到的文本数据
df_new['joined_text']

0       pharaohs leaders believe gods human form . pri...
1       egyptian system government structure like pyra...
2       would use every single piece meat even moldy ....
3       meat spoil , factory would meat chop sausage ,...
4       complex excite pity fear , man good bad misfor...
                              ...                        
1428    `` jonas tell meat take pickle would often fin...
1429    would rub soda take away smell , sell eat free...
1430    one element neutral character make dumb mistak...
1431    various ways factory would cover spoil meat ru...
1432    perfect tragedy , see arrange simple complex p...
Name: joined_text, Length: 1433, dtype: object

In [22]:
# 生成TF-IDF矩阵，注意这里使用的是tfidf_vectorizer.transform而不是fit_transform,用的是之前定义的tfidf_vectorizer实例，不要重新fit！

# 使用同一个 tfidf_vectorizer 来转换验证数据
tfidf_matrix_new = tfidf_vectorizer.transform(df_new['joined_text'])

# 转换为DataFrame，注意这里转换的矩阵是validation_tfidf_matrix
tfidf_df_new = pd.DataFrame(
    tfidf_matrix_new.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)


tfidf_df_new

,10,11,11 12,11 say,12,13,13 say,1906,1967,200,...,write wish,writer,wrong,year,yearly,yearly harvest,yet,yet spite,yet whose,young
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1428,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1429,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1430,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1431,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
X_new = pd.concat([df_new['article_id'],df_features_new, tfidf_df_new], axis=1)
X_new

,article_id,num_chars,num_words,num_sentences,avg_word_length,avg_sentence_length,unique_word_ratio,flesch_reading_ease,gunning_fog,smog_index,...,write wish,writer,wrong,year,yearly,yearly harvest,yet,yet spite,yet whose,young
0,3b9047,529,96,7,5.510417,13.714286,0.604167,75.510526,8.375940,9.236283,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3b9047,343,60,5,5.716667,12.000000,0.600000,64.710000,11.333333,12.602618,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ebad26,234,48,4,4.875000,12.000000,0.729167,76.032500,8.900000,9.725611,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ebad26,480,88,3,5.454545,29.333333,0.704545,54.968485,14.460606,12.457976,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,39c16e,184,36,1,5.111111,36.000000,0.750000,64.545000,15.511111,8.841846,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1428,ebad26,205,41,2,5.000000,20.500000,0.829268,62.049268,16.400000,3.129100,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1429,ebad26,275,53,1,5.188679,53.000000,0.735849,38.111698,22.709434,11.208143,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1430,39c16e,282,53,3,5.320755,17.666667,0.698113,67.590126,10.840252,11.208143,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1431,ebad26,688,130,5,5.292308,26.000000,0.661538,68.512692,12.553846,11.208143,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
X_new = pd.get_dummies(X_new, columns=['article_id'])
X_new

,num_chars,num_words,num_sentences,avg_word_length,avg_sentence_length,unique_word_ratio,flesch_reading_ease,gunning_fog,smog_index,automated_readability_index,...,yearly,yearly harvest,yet,yet spite,yet whose,young,article_id_39c16e,article_id_3b9047,article_id_814d6b,article_id_ebad26
0,529,96,7,5.510417,13.714286,0.604167,75.510526,8.375940,9.236283,6.648839,...,0.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False
1,343,60,5,5.716667,12.000000,0.600000,64.710000,11.333333,12.602618,7.971500,...,0.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False
2,234,48,4,4.875000,12.000000,0.729167,76.032500,8.900000,9.725611,4.723125,...,0.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True
3,480,88,3,5.454545,29.333333,0.704545,54.968485,14.460606,12.457976,14.217576,...,0.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True
4,184,36,1,5.111111,36.000000,0.750000,64.545000,15.511111,8.841846,16.064167,...,0.0,0.0,0.0,0.0,0.0,0.0,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1428,205,41,2,5.000000,20.500000,0.829268,62.049268,16.400000,3.129100,17.795122,...,0.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True
1429,275,53,1,5.188679,53.000000,0.735849,38.111698,22.709434,11.208143,24.887547,...,0.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True
1430,282,53,3,5.320755,17.666667,0.698113,67.590126,10.840252,11.208143,7.842956,...,0.0,0.0,0.0,0.0,0.0,0.0,True,False,False,False
1431,688,130,5,5.292308,26.000000,0.661538,68.512692,12.553846,11.208143,11.823000,...,0.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True


# 做最终的预测

In [25]:
# --- 在新数据集 X_new 上进行预测 ---
print("\n--- 开始在测试集 df_new 上进行预测 ---")

# 重要提示：在调用 .predict() 之前，请务必确认 X_new 的特征列与训练数据 X 的列是完全一致的（名称、顺序和数量）。
# 如果不一致，模型会报错或产生错误的预测结果。
if list(X.columns) != list(X_new.columns):
    print("警告：训练集和测试集的特征列不匹配！请在预测前进行处理。")
else:
    # 使用训练好的最终模型进行预测
    test_predictions = final_model.predict(X_new)

    print("预测完成。")
    print("预测结果的前5个值:")
    print(df_new)
    print(test_predictions[:5])

    # ------------------------------------------------------------------------------------
    # 3. 保存预测结果 (Save Predictions)
    # ------------------------------------------------------------------------------------
    # 通常，我们会将预测结果保存到一个文件中，例如CSV文件，用于提交或后续分析。
    submission_df = pd.DataFrame()
    # 如果df_new有ID列，可以一并加入
    submission_df['student_id'] = df_new['student_id']
    submission_df['article_id'] = df_new['article_id']
    submission_df['score'] = test_predictions

    # 保存到CSV文件
    submission_df.to_csv('results.csv', index=False)

    print("\n预测结果已保存到 'results.csv'")


--- 开始在测试集 df_new 上进行预测 ---
预测完成。
预测结果的前5个值:
        student_id article_id  \
0     3da59a1b970f     3b9047   
1     112abaf5917d     3b9047   
2     e87808d11c34     ebad26   
3     cac8623cbb88     ebad26   
4     f4600c21b673     39c16e   
...            ...        ...   
1428  5ef094caaef7     ebad26   
1429  aff3f12b97d8     ebad26   
1430  0a8f4bedad26     39c16e   
1431  62d75c112422     ebad26   
1432  580d2857420a     39c16e   

                                           summary_text  \
0     The Pharaohs were leaders or believed to be go...   
1          The Egyptian system of government was str...   
2     They would use every single piece of meat they...   
3     When meat spoiled, the factory would can the m...   
4     It should be complex that excites pity and fea...   
...                                                 ...   
1428  "Jonas had told them how the meat that was tak...   
1429  they would rub it up with soda to take away th...   
1430  One element should b